# Amazon Reviews Quality Lakehouse - Walkthrough

This notebook helps you explore Bronze → Silver → Gold outputs after running the pipeline.

## 1) Setup Spark session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("amazon-reviews-quality-lakehouse-walkthrough")
    .getOrCreate()
)

## 2) Define paths

In [ ]:
BRONZE_PATH = "data/bronze"
SILVER_PATH = "data/silver"
QUARANTINE_PATH = "data/quarantine"
GOLD_PATH = "data/gold"

## 3) Read Bronze, Silver, Quarantine

In [ ]:
bronze_df = spark.read.parquet(BRONZE_PATH)
silver_df = spark.read.parquet(SILVER_PATH)
quarantine_df = spark.read.parquet(QUARANTINE_PATH)

print("Bronze count:", bronze_df.count())
print("Silver count:", silver_df.count())
print("Quarantine count:", quarantine_df.count())

In [ ]:
silver_df.select(
    "review_id", "product_id", "star_rating", "review_date",
    "sentiment_label", "sentiment_score", "keyword_theme"
).show(10, truncate=False)

## 4) Quarantine diagnostics

In [ ]:
quarantine_df.groupBy("rejection_reason").count().orderBy(F.desc("count")).show(truncate=False)

## 5) Load Gold marts

In [ ]:
fact_reviews = spark.read.parquet(f"{GOLD_PATH}/fact_reviews")
dim_products = spark.read.parquet(f"{GOLD_PATH}/dim_products")
daily_metrics = spark.read.parquet(f"{GOLD_PATH}/daily_review_metrics")
category_summary = spark.read.parquet(f"{GOLD_PATH}/category_sentiment_summary")
product_summary = spark.read.parquet(f"{GOLD_PATH}/product_review_summary")
negative_theme = spark.read.parquet(f"{GOLD_PATH}/negative_theme_summary")

In [ ]:
print("fact_reviews", fact_reviews.count())
print("dim_products", dim_products.count())
print("daily_metrics", daily_metrics.count())
print("category_summary", category_summary.count())
print("product_summary", product_summary.count())
print("negative_theme", negative_theme.count())

## 6) Quick analytics checks

In [ ]:
category_summary.orderBy(F.desc("avg_sentiment_score")).show(20, truncate=False)
product_summary.orderBy(F.desc("total_reviews")).show(20, truncate=False)
daily_metrics.orderBy("review_date").show(20, truncate=False)

## 7) Zero-null guarantee validation

In [ ]:
critical_cols = ["review_id", "product_id", "review_date", "category", "star_rating", "sentiment_label", "sentiment_score"]
null_checks = [F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in critical_cols]
fact_reviews.select(*null_checks).show(truncate=False)

## 8) Cleanup

In [ ]:
spark.stop()